In [1]:
import os
import glob
from collections import Counter

import laspy
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import StandardScaler

# -----------------------------------------------------
# Einstellungen
# -----------------------------------------------------
INPUT_FOLDER    = "C:/Users/flori/Desktop/pa-1_data/training_2"
TFRECORD_DIR    = "C:/Users/flori/Desktop/pa-1_data/tfrecords"
N_POINTS        = 1024          # Anzahl Punkte pro Chunk
FEATURE_DIM     = 8             # x, y, z, intensity, return_number, number_of_returns, z-min, scan_angle
CHUNKS_PER_SHARD = 1000         # Anzahl Chunks pro TFRecord-Datei

os.makedirs(TFRECORD_DIR, exist_ok=True)

# -----------------------------------------------------
# 1) Scaler anlernen + Label-Statistiken sammeln
# -----------------------------------------------------
scaler = StandardScaler()
label_counter = Counter()
label_set     = set()

file_paths = glob.glob(os.path.join(INPUT_FOLDER, "*.laz"))
print(f"Found {len(file_paths)} .laz files in {INPUT_FOLDER}")

for path in file_paths:
    las = laspy.read(path)
    cls_raw = np.array(las.classification, dtype=np.uint8)
    cls      = np.bitwise_and(cls_raw, 0x1F)

    # Maske: alle Klassen außer Fehlerklasse 7
    valid_mask = cls != 7
    cls_valid  = cls[valid_mask]

    # Labels sammeln und zählen
    label_set.update(np.unique(cls_valid))
    label_counter.update(cls_valid.tolist())

    # Features vorbereiten und Scaler anpassen
    feats = np.vstack((
        las.x, las.y, las.z,
        las.intensity,
        las.return_number,
        las.number_of_returns,
        las.z - np.min(las.z),
        las.scan_angle
    )).T.astype(np.float32)[valid_mask]
    scaler.partial_fit(feats)

print("Scaler stats collected.")
unique_labels = np.array(sorted(label_set), dtype=np.uint8)
# Falls Klasse 7 noch dabei, entfernen (sollte nicht vorkommen)
unique_labels = unique_labels[unique_labels != 7]
label_map = {lab: idx for idx, lab in enumerate(unique_labels)}
num_classes = len(unique_labels)
total_points = sum(label_counter.values())
# Klassengewichte = total_points / (num_classes * class_count)
class_weights = {
    label_map[lab]: total_points / (num_classes * count)
    for lab, count in label_counter.items() if lab in label_map
}
print(f"Detected {num_classes} classes: {unique_labels.tolist()}")
print(f"Class weights: {class_weights}")

# -----------------------------------------------------
# 2) Hilfsfunktion für TFRecord Example
# -----------------------------------------------------
def to_example(features: np.ndarray, labels: np.ndarray, weights: np.ndarray) -> tf.train.Example:
    """
    Packt einen Chunk in ein tf.train.Example.
    - features: [N_POINTS, FEATURE_DIM]
    - labels:   [N_POINTS]
    - weights:  [N_POINTS] (Sample-Weights)
    """
    feat_flat = features.astype(np.float32).reshape(-1).tolist()
    label_flat = labels.astype(np.int64).tolist()
    weight_flat = weights.astype(np.float32).tolist()

    feature = {
        'features': tf.train.Feature(float_list=tf.train.FloatList(value=feat_flat)),
        'labels':   tf.train.Feature(int64_list=tf.train.Int64List(value=label_flat)),
        'weights':  tf.train.Feature(float_list=tf.train.FloatList(value=weight_flat)),
    }
    return tf.train.Example(features=tf.train.Features(feature=feature))

# -----------------------------------------------------
# 3) TFRecord-Schreiber initialisieren
# -----------------------------------------------------
shard_idx = 0
chunk_idx = 0
writer = tf.io.TFRecordWriter(
    os.path.join(TFRECORD_DIR, f"shard_{shard_idx:04d}.tfrecord")
)

# -----------------------------------------------------
# 4) Alle .laz files einlesen und in TFRecords schreiben
# -----------------------------------------------------
for path in file_paths:
    las = laspy.read(path)
    cls_raw = np.array(las.classification, dtype=np.uint8)
    cls      = np.bitwise_and(cls_raw, 0x1F)

    # Maske: nur gültige Klassen
    mask_valid = np.isin(cls, unique_labels)
    if not np.all(cls[~mask_valid] == 7):
        bad = np.setdiff1d(cls, list(unique_labels) + [7])
        if bad.size:
            print(f"⚠️ Unknown codes in {os.path.basename(path)}: {bad}")

    # Roh-Features
    feats = np.vstack((
        las.x, las.y, las.z,
        las.intensity,
        las.return_number,
        las.number_of_returns,
        las.z - np.min(las.z),
        las.scan_angle
    )).T.astype(np.float32)[mask_valid]

    # Labels mappen
    labels = np.array([ label_map[c] for c in cls[mask_valid] ], dtype=np.int64)

    # Normieren
    feats = scaler.transform(feats)
    # Chunking
    num_chunks = feats.shape[0] // N_POINTS
    if num_chunks == 0:
        continue
    feats = feats[:num_chunks * N_POINTS].reshape(num_chunks, N_POINTS, FEATURE_DIM)
    labels = labels[:num_chunks * N_POINTS].reshape(num_chunks, N_POINTS)
    # Zentroid abziehen (nur XYZ, Indizes 0:3)
    centroids = feats[:, :, :3].mean(axis=1, keepdims=True)
    feats[:, :, :3] -= centroids

    # Pro Chunk ein tf.train.Example schreiben
    for i in range(num_chunks):
        X_chunk = feats[i]
        y_chunk = labels[i]
        # Sample-Weights pro Punkt aus class_weights
        w_chunk = np.vectorize(lambda lbl: class_weights[lbl])(y_chunk)

        ex = to_example(X_chunk, y_chunk, w_chunk)
        writer.write(ex.SerializeToString())
        chunk_idx += 1

        # Neues Shard beginnen, wenn CHUNKS_PER_SHARD erreicht
        if chunk_idx % CHUNKS_PER_SHARD == 0:
            writer.close()
            shard_idx += 1
            writer = tf.io.TFRecordWriter(
                os.path.join(TFRECORD_DIR, f"shard_{shard_idx:04d}.tfrecord")
            )

# Schließe letzten Writer
writer.close()
print(f"Wrote {chunk_idx} chunks into {shard_idx+1} TFRecord shards.")


Found 10 .laz files in C:/Users/flori/Desktop/pa-1_data/training_2
Scaler stats collected.
Detected 8 classes: [1, 2, 3, 4, 5, 6, 15, 17]
Class weights: {0: 8.615823257177837, 2: 8.075098927768172, 1: 0.21561738787578455, 4: 0.9460956981799344, 3: 3.079934772264058, 5: 0.5795253742445116, 6: 513.564410210833, 7: 76.35211815572536}
Wrote 81254 chunks into 82 TFRecord shards.


In [2]:
PI = tf.constant(np.pi, dtype=tf.float32)

def random_rotate_pointcloud(batch):
    B      = tf.shape(batch)[0]
    angles = tf.random.uniform((B,), 0.0, 2.0*PI)
    c, s   = tf.cos(angles), tf.sin(angles)
    zeros  = tf.zeros_like(c)
    ones   = tf.ones_like(c)
    R = tf.stack([
        tf.stack([c, -s, zeros], axis=1),
        tf.stack([s,  c, zeros], axis=1),
        tf.stack([zeros, zeros, ones], axis=1),
    ], axis=1)  # (B,3,3)

    xyz  = batch[..., :3]
    rest = batch[..., 3:]
    xyz  = tf.matmul(xyz, R)
    return tf.concat([xyz, rest], axis=-1)

def jitter_pointcloud(batch, sigma=0.01, clip=0.05):
    noise = tf.clip_by_value(
        sigma * tf.random.normal(tf.shape(batch)),
        -clip, clip
    )
    return batch + noise

def scale_pointcloud(batch, scale_low=0.9, scale_high=1.1):
    B      = tf.shape(batch)[0]
    scales = tf.random.uniform((B,1,1), scale_low, scale_high)
    xyz    = batch[..., :3] * scales
    rest   = batch[..., 3:]
    return tf.concat([xyz, rest], axis=-1)

def dropout_pointcloud(batch, keep_prob=0.9):
    mask = tf.cast(
        tf.random.uniform(tf.shape(batch[...,0])) < keep_prob,
        tf.float32
    )
    mask = tf.expand_dims(mask, -1)
    return batch * mask

def augment_fn(X, y):
    X = random_rotate_pointcloud(X)
    X = jitter_pointcloud(X)
    X = scale_pointcloud(X)
    X = dropout_pointcloud(X)
    return X, y

In [3]:
# 1) Parameter
N_POINTS      = 1024
FEATURE_DIM   = 8
BATCH_SIZE    = 64
SHUFFLE_BUF   = 2048
AUTOTUNE      = tf.data.AUTOTUNE
TFRECORD_DIR  = "C:/Users/flori/Desktop/pa-1_data/tfrecords"

# 2) Parser für TFRecord
feature_description = {
    'features': tf.io.FixedLenFeature([N_POINTS * FEATURE_DIM], tf.float32),
    'labels':   tf.io.FixedLenFeature([N_POINTS],           tf.int64),
    'weights':  tf.io.FixedLenFeature([N_POINTS],           tf.float32),
}

def _parse(proto):
    parsed = tf.io.parse_single_example(proto, feature_description)
    X = tf.reshape(parsed['features'], [N_POINTS, FEATURE_DIM])
    y = tf.cast(parsed['labels'], tf.int32)
    w = parsed['weights']  # float32
    return X, y, w

# 3) Dataset aufbauen
files = tf.data.Dataset.list_files(f"{TFRECORD_DIR}/*.tfrecord")
train_ds = (
    files
    .interleave(tf.data.TFRecordDataset, cycle_length=4, num_parallel_calls=AUTOTUNE)
    .map(_parse, num_parallel_calls=AUTOTUNE)
    .shuffle(SHUFFLE_BUF)
    .batch(BATCH_SIZE)
    .map(lambda X, y, w: (augment_fn(X, y)[0], y, w), num_parallel_calls=AUTOTUNE)
    .prefetch(AUTOTUNE)
)

In [4]:
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks

# 1) Modelldefinition (dein PointNet)
def build_pointnet(num_points, num_features, num_classes):
    inputs = layers.Input((num_points, num_features))
    x = layers.Conv1D(64,1,activation='relu')(inputs)
    x = layers.Conv1D(128,1,activation='relu')(x)
    x = layers.Conv1D(1024,1,activation='relu')(x)
    x = layers.Dropout(0.3)(x)

    global_feat = layers.GlobalMaxPooling1D()(x)
    global_feat = layers.RepeatVector(num_points)(global_feat)
    x = layers.Concatenate()([x, global_feat])

    for filters in [512,512,256,128]:
        x = layers.Conv1D(filters,1,activation='relu',
                          kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.3)(x)

    # letzte Schicht mit Softmax
    outputs = layers.Conv1D(num_classes, 1, activation='softmax')(x)
    return models.Model(inputs, outputs)

NUM_CLASSES = num_classes  # aus deinem Preprocessing
model = build_pointnet(N_POINTS, FEATURE_DIM, NUM_CLASSES)

# 2) Kompilieren
# Da wir softmax am Ende haben, setzen wir from_logits=False
model.compile(
    optimizer=optimizers.Adam(5e-4),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=['accuracy']
)

# 3) Callbacks
reduce_lr = callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1
)
early_stop = callbacks.EarlyStopping(
    monitor='val_loss', patience=10, restore_best_weights=True
)
checkpoint = callbacks.ModelCheckpoint(
    "best_pointnet.keras", monitor='val_loss', save_best_only=True, verbose=1
)

# 4) Validation-Datensatz (ohne Gewichtungs-Map, nur X,y)
val_ds = (
    tf.data.Dataset.list_files(f"{TFRECORD_DIR}/*.tfrecord")
    .interleave(tf.data.TFRecordDataset, cycle_length=4, num_parallel_calls=tf.data.AUTOTUNE)
    .map(_parse, num_parallel_calls=tf.data.AUTOTUNE)      # liefert X,y,w
    .map(lambda X,y,w: (X,y), num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

# 5) Training
history = model.fit(
    train_ds,                       # liefert X,y,w → Keras interpretiert w als sample_weight
    validation_data=val_ds,         # liefert X,y
    epochs=50,
    steps_per_epoch=(chunk_idx // BATCH_SIZE),
    callbacks=[reduce_lr, early_stop, checkpoint]
)


Epoch 1/50
  10/1269 ━━━━━━━━━━━━━━━━━━━━ 54:42 3s/step - accuracy: 0.1678 - loss: 3.2874

KeyboardInterrupt: 